Silver: cleaned jobs (trim, null text → NULL, remote → boolean, one row per job)

In [0]:
dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date")

bronze_rows = dbutils.jobs.taskValues.get(
    taskKey="bronze", key="bronze_rows", debugValue=0
)
print(f"Running silver for: {run_date or 'all data'}")
print(f"Bronze loaded {bronze_rows} rows")

Running silver for: all data
Bronze loaded 0 rows


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("workspace.default.bronze_job_postings")

def clean(col_name):
    # trim spaces; turn "" and "null" into a real NULL
    c = F.trim(F.col(col_name))
    return F.when(F.lower(c).isin("", "null"), None).otherwise(c)

jobs = (bronze
        .withColumn("job_id",    clean("job_id"))
        .withColumn("title",     clean("title"))
        .withColumn("company",   clean("company"))
        .withColumn("location",  clean("location"))
        .withColumn("work_type", clean("work_type"))
        .withColumn("url",       clean("url"))
        .withColumn("remote",    F.lower(clean("remote")) == "true")   # text "TRUE"/"FALSE" -> boolean
        .withColumn("has_company", F.col("company").isNotNull()))

# one row per job_id, keeping the latest load
w = Window.partitionBy("job_id").orderBy(F.col("_ingested_at").desc())
silver = (jobs
          .withColumn("rn", F.row_number().over(w))
          .filter("rn = 1")
          .drop("rn")
          .withColumn("_processed_at", F.current_timestamp()))

silver.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
      .saveAsTable("workspace.default.silver_jobs")

In [0]:
%sql
SELECT COUNT(*)                   AS total_rows,      -- 804
       COUNT(DISTINCT job_id)     AS distinct_jobs,   -- 804
       COUNT_IF(company IS NULL)  AS no_company,      -- 9 (real NULLs now)
       COUNT_IF(remote)           AS remote_jobs,
       COUNT_IF(NOT remote)       AS onsite_or_hybrid
FROM silver_jobs;

total_rows,distinct_jobs,no_company,remote_jobs,onsite_or_hybrid
804,804,9,241,563


Bronze vs. silver reconciliation

In [0]:
%sql
SELECT
  COUNT(*)                                                    AS jobs_compared,
  COUNT_IF(b.title    IS DISTINCT FROM s.title)               AS title_changed,
  COUNT_IF(b.company  IS DISTINCT FROM s.company)             AS company_changed,
  COUNT_IF(b.location IS DISTINCT FROM s.location)            AS location_changed,
  COUNT_IF(b.work_type IS DISTINCT FROM s.work_type)          AS work_type_changed,
  COUNT_IF(b.url      IS DISTINCT FROM s.url)                 AS url_changed
FROM bronze_job_postings b
JOIN silver_jobs s ON trim(b.job_id) = s.job_id;

jobs_compared,title_changed,company_changed,location_changed,work_type_changed,url_changed
804,28,21,0,0,0
